In [ ]:
#HOUDEN!!!

import numpy as np
from mcap.reader import make_reader
from mcap_ros2.decoder import Decoder
import struct

def extract_merged_lidar_frame(mcap_path, output_bin, target_timestamp_float, lidar_topics):
    target_ns = int(target_timestamp_float * 1e9)
    # We zoeken nu in een ruimer venster van 0.5 seconden rond de tijd
    start_search = target_ns - 250_000_000 
    end_search = target_ns + 250_000_000

    all_points_list = []

    with open(mcap_path, "rb") as f:
        reader = make_reader(f)
        decoder = Decoder()
        
        for topic in lidar_topics:
            print(f"Zoeken naar dichtstbijzijnde frame voor {topic}...")
            best_msg = None
            min_diff = float('inf')
            best_schema = None

            # Stap 1: Zoek het bericht dat qua tijd het dichtst bij target_ns ligt
            for schema, channel, message in reader.iter_messages(
                topics=[topic], 
                start_time=start_search, 
                end_time=end_search
            ):
                diff = abs(message.log_time - target_ns)
                if diff < min_diff:
                    min_diff = diff
                    best_msg = message
                    best_schema = schema

            if best_msg:
                # Stap 2: Decodeer en extraheer het beste bericht
                try:
                    ros_msg = decoder.decode(best_schema, best_msg)
                except:
                    ros_msg = decoder.decode(best_msg)
                
                print(f"-> Gevonden op {best_msg.log_time / 1e9} (verschil: {min_diff/1e6:.1f}ms)")
                
                point_step = ros_msg.point_step
                num_points = ros_msg.width * ros_msg.height
                data = ros_msg.data
                topic_points = np.zeros((num_points, 5), dtype=np.float32)
                
                # Betrouwbare extractie
                for i in range(num_points):
                    offset = i * point_step
                    try:
                        # x, y, z, intensity (floats)
                        pt = struct.unpack_from('ffff', data, offset)
                        topic_points[i, 0:4] = pt
                        # Ring index (proberen op offset 16)
                        ring = struct.unpack_from('H', data, offset + 16)[0]
                        topic_points[i, 4] = float(ring)
                    except:
                        continue
                
                all_points_list.append(topic_points)
                found_topic = True
            else:
                print(f"-> WAARSCHUWING: Helemaal geen data gevonden voor {topic}")

    if all_points_list:
        final_merged_points = np.vstack(all_points_list)
        final_merged_points.tofile(output_bin)
        print(f"\nSucces! Totaal {len(final_merged_points)} punten opgeslagen in:\n{output_bin}")
    else:
        print("Geen data gevonden.")

# --- INSTELLINGEN ---
MCAP_FILE = r"C:\rosbag_0.mcap"
OUTPUT_FILE = r"C:\Us   ers\Lars Wissink\OneDrive\Documenten\lars wissink\WB TU Delft\jaar 3\bep\.mcap to Bin downloads\merged_full_scan.bin"
TARGET_TIME = 1762960978.527071919

TOPICS_TO_MERGE = [
    "/rslidar/M1P_deskewed",
    "/rslidar/helios_L",
    "/rslidar/helios_R"
]

extract_merged_lidar_frame(MCAP_FILE, OUTPUT_FILE, TARGET_TIME, TOPICS_TO_MERGE)

In [ ]:
#houden!!!
import open3d as o3d
import numpy as np
import time

bin_path = r"C:\Users\Lars Wissink\OneDrive\Documenten\lars wissink\WB TU Delft\jaar 3\bep\.mcap to Bin downloads\gps_final_test.bin"
print("1. Bestand laden...")
data = np.fromfile(bin_path, dtype=np.float32).reshape(-1, 5)
initial_count = len(data)

# STAP 1: Verwijder alle rijen die NaN bevatten in de eerste 3 kolommen (XYZ)
print("2. Ongeldige punten (NaN) verwijderen...")
mask = ~np.isnan(data[:, :3]).any(axis=1)
clean_data = data[mask]

# STAP 2: Verwijder punten die op (0,0,0) staan (vaak ook ruis)
mask_zero = ~((clean_data[:, 0] == 0) & (clean_data[:, 1] == 0) & (clean_data[:, 2] == 0))
clean_data = clean_data[mask_zero]

final_count = len(clean_data)
print(f"   -> Gefilterd: {initial_count - final_count} punten verwijderd.")
print(f"   -> Overgebleven: {final_count} punten.")

# STAP 3: Omzetten naar Open3D
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(clean_data[:, :3])

# Kleuren op basis van intensiteit (kolom 4)
intensities = clean_data[:, 3]
# Normaliseer intensiteit voor kleuren (0 tot 1)
if intensities.max() > 0:
    colors = intensities / intensities.max()
    # Maak een grijswaarden-map (RGB waarbij R=G=B)
    rgb_colors = np.stack([colors, colors, colors], axis=1)
    pcd.colors = o3d.utility.Vector3dVector(rgb_colors)

# STAP 4: Downsampling (nu zonder hangen!)
print("3. Voxel downsampling (0.1m)...")
pcd = pcd.voxel_down_sample(voxel_size=0.1)

# STAP 5: Tonen
print("4. Visualisatie openen...")
o3d.visualization.draw_geometries([pcd], 
                                  window_name=f"Cleaned Cloud - {final_count} pts",
                                  width=1200, height=800)

In [ ]:
import numpy as np
import struct
from mcap.reader import make_reader
from mcap_ros2.decoder import Decoder
from scipy.spatial.transform import Rotation as R

# --- STEL HIER DE CORRECTIE IN ---
# Probeer 90.0, -90.0, of 180.0 als de kaart verdraaid blijft
YAW_OFFSET_DEGREES = 90.0 

def get_relative_meters(lat, lon, start_lat, start_lon):
    y = (lat - start_lat) * 111319.9
    x = (lon - start_lon) * (111319.9 * np.cos(np.radians(start_lat)))
    return x, y

def extract_aligned_gps(mcap_path, output_bin, start_time, end_time, step_seconds):
    start_ns, end_ns = int(start_time * 1e9), int(end_time * 1e9)
    step_ns = int(step_seconds * 1e9)
    
    topics = ["/rslidar/M1P_deskewed", "/rslidar/helios_L", "/rslidar/helios_R"]
    odom_topic, gps_topic = "/odometry/global", "/gps/filtered"
    
    all_points = []
    start_gps = None
    # Maak de correctie-rotatie aan
    correction_rot = R.from_euler('z', YAW_OFFSET_DEGREES, degrees=True)

    with open(mcap_path, "rb") as f:
        reader = make_reader(f)
        decoder = Decoder()
        
        for target_ns in range(start_ns, end_ns, step_ns):
            curr_gps = next((decoder.decode(s, m) for s, c, m in reader.iter_messages(topics=[gps_topic], start_time=target_ns-500_000_000, end_time=target_ns+500_000_000)), None)
            curr_odom = next((decoder.decode(s, m) for s, c, m in reader.iter_messages(topics=[odom_topic], start_time=target_ns-200_000_000, end_time=target_ns+200_000_000)), None)

            if not curr_gps or not curr_odom: continue
            if start_gps is None: start_gps = (curr_gps.latitude, curr_gps.longitude)

            dx, dy = get_relative_meters(curr_gps.latitude, curr_gps.longitude, start_gps[0], start_gps[1])
            
            # Huidige rotatie van de auto + onze handmatige correctie
            q = curr_odom.pose.pose.orientation
            current_rot = R.from_quat([q.x, q.y, q.z, q.w]) * correction_rot

            for t_name in topics:
                for s, c, m in reader.iter_messages(topics=[t_name], start_time=target_ns-100_000_000, end_time=target_ns+100_000_000):
                    msg = decoder.decode(s, m)
                    p_data, step = msg.data, msg.point_step
                    
                    for i in range(msg.width * msg.height):
                        off = i * step
                        try:
                            raw = struct.unpack_from('ffff', p_data, off)
                            if np.isnan(raw[0]) or (abs(raw[0]) < 1.2 and abs(raw[1]) < 1.2): continue
                            
                            # Transformeer punt
                            p_local = np.array([raw[0], raw[1], raw[2]])
                            p_rotated = current_rot.apply(p_local)
                            
                            # Voeg GPS meters toe
                            all_points.append([p_rotated[0] + dx, p_rotated[1] + dy, p_rotated[2], raw[3], 0.0])
                        except: continue
                    break
            print(f"Verwerkt: {dx:.1f}m vooruit...")

    if all_points:
        np.array(all_points, dtype=np.float32).tofile(output_bin)
        print(f"Klaar! Opgeslagen in {output_bin}")

# --- RUN ---
extract_aligned_gps(r"C:\rosbag_0.mcap", r"C:\Users\Lars Wissink\OneDrive\Documenten\lars wissink\WB TU Delft\jaar 3\bep\.mcap to Bin downloads\yaw_corrected.bin", 1762960978.58, 1762960986.98, 0.2)

In [ ]:
import numpy as np
import struct
from mcap.reader import make_reader
from mcap_ros2.decoder import Decoder
from scipy.spatial.transform import Rotation as R

def get_relative_meters(lat, lon, start_lat, start_lon):
    """Rekent GPS coördinaten om naar meters t.o.v. het startpunt."""
    y = (lat - start_lat) * 111319.9
    x = (lon - start_lon) * (111319.9 * np.cos(np.radians(start_lat)))
    return x, y

def extract_imu_aligned(mcap_path, output_bin, start_time, end_time, step_seconds):
    start_ns, end_ns = int(start_time * 1e9), int(end_time * 1e9)
    step_ns = int(step_seconds * 1e9)
    
    # Topics gebaseerd op jouw screenshot (WhatsApp Image 2026-04-30 at 13.56.46_4.jpeg)
    topics = ["/rslidar/M1P_deskewed", "/rslidar/helios_L", "/rslidar/helios_R"]
    imu_topic = "/imu/data" 
    gps_topic = "/gps/filtered"
    
    all_points = []
    start_gps = None
    start_rot_inv = None

    print(f"Start met verwerken van {mcap_path}...")

    with open(mcap_path, "rb") as f:
        reader = make_reader(f)
        decoder = Decoder()
        
        for target_ns in range(start_ns, end_ns, step_ns):
            # Vergroot de marge naar 1 seconde (1_000_000_000 ns) voor meer kans op een match
            marge = 1_000_000_000 
            
            curr_gps = next((decoder.decode(s, m) for s, c, m in reader.iter_messages(topics=[gps_topic], start_time=target_ns-marge, end_time=target_ns+marge)), None)
            curr_imu = next((decoder.decode(s, m) for s, c, m in reader.iter_messages(topics=[imu_topic], start_time=target_ns-marge, end_time=target_ns+marge)), None)

            if not curr_gps:
                print(f"Sla over: Geen GPS gevonden rond {target_ns/1e9:.2f}")
                continue
            if not curr_imu:
                print(f"Sla over: Geen IMU gevonden rond {target_ns/1e9:.2f}")
                continue

        

            # Nulmeting vastleggen
            if start_gps is None:
                start_gps = (curr_gps.latitude, curr_gps.longitude)
                q = curr_imu.orientation
                start_rot_inv = R.from_quat([q.x, q.y, q.z, q.w]).inv()
                print(f"Referentiepunt gezet op GPS: {start_gps}")

            # 2. Bereken de huidige positie (meters) en rotatie (IMU)
            dx, dy = get_relative_meters(curr_gps.latitude, curr_gps.longitude, start_gps[0], start_gps[1])
            
            q = curr_imu.orientation
            rel_rot = start_rot_inv * R.from_quat([q.x, q.y, q.z, q.w])
            offset_pos = start_rot_inv.apply([dx, dy, 0])

            # 3. LiDAR punten inladen en transformeren
            for t_name in topics:
                for s, c, m in reader.iter_messages(topics=[t_name], start_time=target_ns-100_000_000, end_time=target_ns+100_000_000):
                    msg = decoder.decode(s, m)
                    p_data, step = msg.data, msg.point_step
                    
                    for i in range(msg.width * msg.height):
                        off = i * step
                        try:
                            # x, y, z, intensity
                            raw = struct.unpack_from('ffff', p_data, off)
                            if np.isnan(raw[0]) or (abs(raw[0]) < 1.2 and abs(raw[1]) < 1.2): continue
                            
                            # Transformatie: Rotatie (IMU) -> Translatie (GPS)
                            p_local = np.array([raw[0], raw[1], raw[2]])
                            p_world = rel_rot.apply(p_local) + offset_pos
                            
                            # Bewaren in lijst (x, y, z, intensity, ring=0)
                            all_points.append([p_world[0], p_world[1], p_world[2], raw[3], 0.0])
                        except: continue
                    break 
            print(f"Timestamp {target_ns/1e9:.2f}: {dx:.2f}m verschoven.")

    # --- HIER WORDT HET BESTAND DAADWERKELIJK OPSLAGEN ---
    if all_points:
        # Omzetten naar float32 voor compatibiliteit met de meeste visualizers
        data_to_save = np.array(all_points, dtype=np.float32)
        data_to_save.tofile(output_bin)
        print(f"\nSucces! Het bestand is aangemaakt:")
        print(f"Locatie: {output_bin}")
        print(f"Totaal aantal punten: {len(all_points)}")
    else:
        print("Fout: Geen punten gevonden. Controleer de timestamps en topics.")

# --- INSTELLINGEN ---
# Zorg dat dit pad klopt en de map bestaat!
LOCATIE_BIN = r"C:\Users\Lars Wissink\OneDrive\Documenten\lars wissink\WB TU Delft\jaar 3\bep\.mcap to Bin downloads\imu_aligned_map.bin"
LOCATIE_MCAP = r"C:\rosbag_0.mcap"

# Uitvoeren
extract_imu_aligned(LOCATIE_MCAP, LOCATIE_BIN, 1762960978.58, 1762960986.98, 0.2)